# 第03章 离散的世界 · C组人工复核

| 被审版本 | 组别 | 提交 SHA | 文件 |
| --- | --- | --- | --- |
| A1 | G3 | 0f1665760fa5460055e982f34dd5ba5d2335e0e2 | 正文、实验、习题及参考答案 |
| A2 | G13 | a38835686d99a2f7940c924390e5593b9997f258 | 正文、实验、习题及参考答案 |
| B1 | G8 | 9f8c3e783adb4b5968681211e0cfaec32556dc0c | 修正清单、验证 |
| B2 | G17 | 7af3d31f086cbf8c8427e101cc7ba6bbe0f2756c | 修正清单、验证 |

## 手工独立推导

### 假设、符号与适用条件

- ODE 步长 $h$；热方程空间步长 $h$、时间步长 $\tau$、$r=\kappa\tau/h^2$；有限元 $[0,1]$ 上 $N$ 个等长区间，$h=1/N$。
- 阶条件按“局部截断误差＝一步缺陷／$h$”定义；全局阶另需右端光滑、一步映射在固定区间有 $(1+Ch)$ 型 Lipschitz 上界。
- 稳定性统一用 2 范数；网格固定的谱阈值、$r\le1/2$（一维周期或无限网格最坏模态）与 $r\le1/2$（局部凸组合）是三个不同口径，不能互换。
- 边界数据按坐标导数 $u_x$ 书写；外法向导数需换号。

### 推导与结论

1. 经典 RK4 的八条阶条件成立，但 A1 §3.1.3.2 的阶段展开在 $h^2b^Tc$ 前多乘了 $1/2$，两个三阶项系数也错；用 $y'=t$ 一步即可分辨（正确 $h^2/2$，原式 $h^2/4$）。这是展开错，不是算法错。
2. 一维齐次 Dirichlet 的精确固定网格门槛为 $r\le[2\cos^2(\pi/(2N))]^{-1}$，与 $r\le1/2$ 不同；$N=5,r=0.51$ 谱半径 $0.845$ 但单点更新已为 $-0.02$，说明“二范数未增”与“不产生非物理振荡”是两件事。
3. 有限元未施加边界时 $\ker K=\operatorname{span}\{\mathbf 1\}$；置 1、消元、置大数给出三个不同的条件数，切片后置大数内部块不变。
4. 残差到误差需要区间积分控制或解析参考；固定参数下两求解器之差只是两个离散误差之差，不是单个离散误差。
5. 固定比值梯度权重 $\lambda_k=10^{-1.8}\lambda_{k-1}$ 没有正平衡点，不能把“三步拉到同量级”当作结论。

## 3.1 ODE初值问题：Euler与RK4（2学时）

| 被审结论 | 验证条件或反例 | 手工结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| A1 §3.1.3.2：由阶段展开列出八条阶条件，RK4 更新正确 | 代入经典 RK4 系数逐条核 $b^Tc^k,b^TA^jc$；再用 $y'=t$ 一步比较两种展开 | 八条条件值 1、1/2、1/3、1/6、1/4、1/8、1/12、1/24 全部成立；原展开多乘 1/2 使 $h^2$ 项减半 | 阶条件全对；$y'=t,h=1/4$ 一步 $1/32$，原式给 $1/64$ | 条件与算法成立，展开式需修正（C1-01） |
| B1 测试结果 3.1：Euler 一阶、RK4 四阶 | $y'=-y,\ y(0)=1$ 到 $t=1$，$N=64\to128$ 观测阶 | $1+h$ 与 $R(z)$ 的 $O(h)$、$O(h^4)$ 结构 | 1.00472864033548、4.009415224860062 | 成立，与 A1 说明一致 |
| A2 表 3.1-7 / G17-B002：等精度成本 4096、4096、8 步 | 固定 $y'=-y,T=2$、容差 $10^{-4}$，逐个候选 $N=2^k$ | RK4/FE 每步右端调用 4:1；总调用比 128:1（若只按显式求值计） | FE 4096（$6.609\times10^{-5}$）、BE 4096（$6.608\times10^{-5}$）、RK4 8（$1.086\times10^{-5}$） | 成立 |
| A2 习题 3.1-3：$y'=1-2(y-t)$ 在 $h=5/4$ 的终点值 | 代换 $w=y-t$ 后三种放大因子 | FE $-3/2$、BE $2/7$、RK4 $83/128$ | 10.062500000、5.006663890、5.176796023 | 成立；增量稳定≠终点误差为零 |

In [1]:
import numpy as np

# --- 1) 经典 RK4 的八条阶条件 ---
A = np.array([[0,0,0,0],[.5,0,0,0],[0,.5,0,0],[0,0,1,0]], float)
b = np.array([1/6,1/3,1/3,1/6])
c = A.sum(1)
print("c   =", c)
print("Ac  =", A@c, " Ac^2 =", A@(c**2), " A^2c =", A@A@c)
conds = {"b^T1": b.sum(), "b^Tc": b@c, "b^Tc^2": b@c**2, "b^TAc": b@(A@c),
         "b^Tc^3": b@c**3, "b^T(c*Ac)": b@(c*(A@c)), "b^TAc^2": b@(A@(c**2)),
         "b^TA^2c": b@(A@A@c)}
target = [1, 1/2, 1/3, 1/6, 1/4, 1/8, 1/12, 1/24]
for k, t in zip(conds, target):
    print(f"{k:10s} = {conds[k]:.12f}  目标 {t:.12f}  差 {abs(conds[k]-t):.1e}")

# --- 2) y'=t 一步：正确更新 vs A1 原展开 ---
def rk4(f, t, y, h):
    k1 = f(t, y); k2 = f(t+h/2, y+h*k1/2); k3 = f(t+h/2, y+h*k2/2); k4 = f(t+h, y+h*k3)
    return y + h*(k1+2*k2+2*k3+k4)/6

h = 0.25
print("\ny'=t, t0=0, y0=0, h=0.25")
print("  RK4 一步 =", rk4(lambda t, y: t, 0.0, 0.0, h), "  h^2/2 =", h**2/2)
print("  原展开 h^2 项系数减半后 =", h**2/4)

# --- 3) 线性衰减算例的观测阶 ---
f = lambda t, y: -y
def step_euler(f, t, y, hh): return y + hh*f(t, y)
def err(method, N):
    y, t, hh = 1.0, 0.0, 1.0/N
    for _ in range(N):
        y = method(f, t, y, hh); t += hh
    return abs(y - np.exp(-1))
for name, m in [("Euler", step_euler), ("RK4", rk4)]:
    e = {N: err(m, N) for N in (64, 128)}
    print(f"  {name}: |e(64)|={e[64]:.6e}, |e(128)|={e[128]:.6e}, 观测阶={np.log2(e[64]/e[128]):.6f}")

c   = [0.  0.5 0.5 1. ]
Ac  = [0.   0.   0.25 0.5 ]  Ac^2 = [0.    0.    0.125 0.25 ]  A^2c = [0.   0.   0.   0.25]
b^T1       = 1.000000000000  目标 1.000000000000  差 1.1e-16
b^Tc       = 0.500000000000  目标 0.500000000000  差 0.0e+00
b^Tc^2     = 0.333333333333  目标 0.333333333333  差 0.0e+00
b^TAc      = 0.166666666667  目标 0.166666666667  差 0.0e+00
b^Tc^3     = 0.250000000000  目标 0.250000000000  差 0.0e+00
b^T(c*Ac)  = 0.125000000000  目标 0.125000000000  差 0.0e+00
b^TAc^2    = 0.083333333333  目标 0.083333333333  差 0.0e+00
b^TA^2c    = 0.041666666667  目标 0.041666666667  差 0.0e+00

y'=t, t0=0, y0=0, h=0.25
  RK4 一步 = 0.03125   h^2/2 = 0.03125
  原展开 h^2 项系数减半后 = 0.015625
  Euler: |e(64)|=2.892917e-03, |e(128)|=1.441725e-03, 观测阶=1.004729
  RK4: |e(64)|=1.851229e-10, |e(128)|=1.149497e-11, 观测阶=4.009409


In [2]:
import numpy as np

# --- 等精度成本：y'=-y, y(0)=1, T=2, 容差 1e-4 ---
truth = np.exp(-2.0)
def e_fe(N): return abs((1 - 2.0/N)**N - truth)
def e_be(N):
    # 隐式 Euler：y_{n+1}=y_n/(1+2/N)
    return abs((1.0/(1 + 2.0/N))**N - truth)
def e_rk(N):
    z = -2.0/N
    return abs((1 + z + z**2/2 + z**3/6 + z**4/24)**N - truth)

print("候选 N=2^k，首次落进 1e-4：")
for name, fn, mult in [("FE", e_fe, 1), ("BE", e_be, 0), ("RK4", e_rk, 4)]:
    prev = None
    for N in [2**k for k in range(1, 15)]:
        e = fn(N)
        if e < 1e-4:
            ops = N*mult if mult else N
            print(f"  {name:4s} 前一档 N={prev[0]} err={prev[1]:.6e} | 首次达标 N={N} err={e:.6e} | 操作 {ops}")
            break
        prev = (N, e)
print("  FE:RK4 总调用比 =", 4096*1/32, "（固定 h 时单步调用比 4:1）")

# --- A2 习题 3.1-3：w=y-t 满足 w'=-2w, w(0)=1 ---
def R4(z): return 1 + z + z*z/2 + z**3/6 + z**4/24
h = 5.0/4
print("\n习题 3.1-3，h=5/4, w0=1")
print("  FE 因子 =", 1 - 2*h, " BE 因子 =", 1/(1 + 2*h), " RK4 因子 =", R4(-2*h))
print("  y(5) = 5 + w(5)： FE %.9f  BE %.9f  RK4 %.9f" % (5 + (1-2*h)**4, 5 + (1/(1+2*h))**4, 5 + R4(-2*h)**4))

候选 N=2^k，首次落进 1e-4：
  FE   前一档 N=2048 err=1.321849e-04 | 首次达标 N=4096 err=6.608706e-05 | 操作 4096
  BE   前一档 N=2048 err=1.321419e-04 | 首次达标 N=4096 err=6.607630e-05 | 操作 4096
  RK4  前一档 N=4 err=2.144873e-04 | 首次达标 N=8 err=1.085872e-05 | 操作 32
  FE:RK4 总调用比 = 128.0 （固定 h 时单步调用比 4:1）

习题 3.1-3，h=5/4, w0=1
  FE 因子 = -1.5  BE 因子 = 0.2857142857142857  RK4 因子 = 0.6484375000000002
  y(5) = 5 + w(5)： FE 10.062500000  BE 5.006663890  RK4 5.176796023


## 3.2 PDE有限差分：热传导与CFL条件（2学时）

| 被审结论 | 验证条件或反例 | 手工结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| A1 §3.2.3 前文与表 10：把 $r\le1/2$ 当作（近乎）通用 CFL 阈值 | 齐次 Dirichlet 的 $B=I+r\,\mathrm{tridiag}(1,-2,1)$，正弦代入 | 有限网格精确门槛 $r\le[2\cos^2(\pi/(2N))]^{-1}$，$r\le1/2$ 只是细网格极限/周期情形 | $N=5$ 门槛 $0.552786$，$r=0.51$ 谱半径 $0.845197$，单点更新 $1-2r=-0.02$ | 前文口径错，后文与表 10 应统一（C1-02） |
| A1 单元 25 / B1-B001：$\max|r_{\rm alg}|=0.16$ | 按原代码保留 $[U_1,\dots,U_4]$ 却套端点行 | 正确首行为 $1.8u_0-2ru_1$，右端含 $-2rh\sigma_L$ | 错误行 $0.04$ 对右端 $0.2$，残差 $-0.16$；正确行 $=-0.16$ 与右端一致，残差 0 | 通量丢失＋编号错位，确认 B001 |
| 边界补丁的装配是否自洽 | 8 个线性精确解 × 4 种边界组合 | 线性解二阶差分为 0，内部行与端点行都应为恒等式 | 最大残差约 $2.22\times10^{-16}$ | 装配正确，可用于后续复测 |
| A2 一维算例 / G17-B001 二维实验 | $J=40$、$r=0.4$ 与 $0.6$；单位正方形五点格式 | 模态 $G_k=1-4r\sin^2(k\pi/2J)$；二维 $g_{pq}=1-8r$（$p=q=N-1$） | $G_{39}=-0.597534,\ -1.396301$；纯模态零边界 $3.7119\times10^{-4}$、$1.2883\times10^{-1}$ | 阈值判断成立；原实验边界未清零，不能直接验零边界阈值 |

In [3]:
import numpy as np

# --- 1) 一维有限网格的精确阈值 ---
N, r = 5, 0.51
tridiag = np.diag(-2*np.ones(N-1)) + np.diag(np.ones(N-2), 1) + np.diag(np.ones(N-2), -1)
B = np.eye(N-1) + r*tridiag
ev = np.linalg.eigvalsh(B)
print("N=5, r=0.51")
print("  特征值 =", np.round(ev, 8), " 谱半径 =", max(abs(ev)))
print("  精确门槛 1/(2cos^2(pi/2N)) =", 1/(2*np.cos(np.pi/(2*N))**2))
print("  单点更新 1-2r =", 1-2*r, "（负值＝非物理振荡）")
print("\n各 N 的有限网格门槛（均严格大于 1/2，故 r<=1/2 充分但不紧）：")
for NN in (5, 10, 20, 40, 80, 160):
    thr = 1/(2*np.cos(np.pi/(2*NN))**2)
    print(f"  N={NN:3d} 门槛={thr:.9f}  与 1/2 之差={thr-0.5:.3e}")

# --- 2) B001：原首行残差 ---
N2, h, r2 = 5, 0.2, 0.4
sigmaL = 1.0
print("\nB001（u=x，左导数 1，右值 1，N=5, h=0.2, r=0.4）")
print("  正确首行 LHS =", (1+2*r2)*0.0 - 2*r2*0.2, " RHS =", 0.0 - 2*r2*h*sigmaL)
print("  原代码首行 LHS =", (1+2*r2)*0.2 - 2*r2*0.4, " RHS = 0.2  -> 残差 =", (1+2*r2)*0.2 - 2*r2*0.4 - 0.2)
print("  正确末行 LHS =", -2*r2*0.8 + (1+2*r2)*1.0, " RHS =", 1.0 + 2*r2*h*sigmaL)

# --- 3) 8 个线性补丁的装配残差（线性精确解 → 二阶差分为零）---
N3, h3, r3 = 6, 1/6, 0.4
def residual(bcL, bcR, u):
    A = np.zeros((N3+1, N3+1)); rhs = np.zeros(N3+1)
    slope_L = (u[1]-u[0])/h3
    slope_R = (u[N3]-u[N3-1])/h3
    for j in range(1, N3):
        A[j, j-1], A[j, j], A[j, j+1] = -r3, 1+2*r3, -r3
        rhs[j] = u[j]                      # 无源，u 线性故二阶差分为 0
    if bcL == 'N':
        A[0, 0], A[0, 1] = 1+2*r3, -2*r3
        rhs[0] = u[0] - 2*r3*h3*slope_L
    else:
        A[0, 0], rhs[0] = 1.0, u[0]
    if bcR == 'N':
        A[N3, N3], A[N3, N3-1] = 1+2*r3, -2*r3
        rhs[N3] = u[N3] + 2*r3*h3*slope_R
    else:
        A[N3, N3], rhs[N3] = 1.0, u[N3]
    return np.max(np.abs(A@u - rhs))

x = np.arange(N3+1)/N3
res = []
for bcL in "ND":
    for bcR in "ND":
        for u in (lambda t: 1.0*t + 0.0, lambda t: 2.0*t + 0.5):
            res.append(residual(bcL, bcR, u(x)))
print(f"\n线性补丁 {len(res)} 个（4 种边界组合 × 2 个线性解），最大装配残差 = {max(res):.3e}")

N=5, r=0.51
  特征值 = [-0.84519733 -0.33519733  0.29519733  0.80519733]  谱半径 = 0.8451973342624463
  精确门槛 1/(2cos^2(pi/2N)) = 0.5527864045000421
  单点更新 1-2r = -0.020000000000000018 （负值＝非物理振荡）

各 N 的有限网格门槛（均严格大于 1/2，故 r<=1/2 充分但不紧）：
  N=  5 门槛=0.552786405  与 1/2 之差=5.279e-02
  N= 10 门槛=0.512542815  与 1/2 之差=1.254e-02
  N= 20 门槛=0.503096979  与 1/2 之差=3.097e-03
  N= 40 门槛=0.500771856  与 1/2 之差=7.719e-04
  N= 80 门槛=0.500192815  与 1/2 之差=1.928e-04
  N=160 门槛=0.500048195  与 1/2 之差=4.819e-05

B001（u=x，左导数 1，右值 1，N=5, h=0.2, r=0.4）
  正确首行 LHS = -0.16000000000000003  RHS = -0.16000000000000003
  原代码首行 LHS = 0.03999999999999998  RHS = 0.2  -> 残差 = -0.16000000000000003
  正确末行 LHS = 1.16  RHS = 1.1600000000000001

线性补丁 8 个（4 种边界组合 × 2 个线性解），最大装配残差 = 4.441e-16


In [4]:
import numpy as np

# --- A2 一维算例：J=40, kappa=0.1, T=0.6 ---
J = 40
for r in (0.4, 0.6):
    G39 = 1 - 4*r*np.sin(39*np.pi/(2*J))**2
    steps = 240 if r == 0.4 else 160
    print(f"r={r}: G_39={G39:.9f}  高频终点振幅={abs(1e-12*G39**steps):.3e}")

# --- 固定 r=0.4 的网格加密（零边界、单模态解析对照）---
r, kap, T = 0.4, 0.1, 0.6
truth = np.exp(-kap*np.pi**2*T)
print("\n固定 r=0.4，终点误差 |G_1^M - e^{-0.06pi^2}|：")
for Jc in (20, 40, 80, 160):
    M = int(round(kap*T*Jc*Jc/r))
    G1 = 1 - 4*r*np.sin(np.pi/(2*Jc))**2
    print(f"  J={Jc:3d} M={M:4d} err={abs(G1**M - truth):.9e}")
    truth_ = truth
    # 与文档一致：
print("  说明：上表与 A2、B2 的对应结果一致至舍入容差。")

# --- 含 1e-12 高频扰动的稳定组实算 ---
J, r, M = 40, 0.4, 240
xg = np.arange(J+1)/J
U = np.sin(np.pi*xg) + 1e-12*np.sin(39*np.pi*xg)
U[0] = U[-1] = 0.0
for _ in range(M):
    U[1:-1] = U[1:-1] + r*(U[2:] - 2*U[1:-1] + U[:-2])
exact = np.sin(np.pi*xg)*np.exp(-kap*np.pi**2*T) + 1e-12*np.sin(39*np.pi*xg)
print(f"  稳定组终点最大误差 = {np.max(np.abs(U-exact)):.6e}（主要来自低频离散误差）")

# --- 二维：有限网格阈值与纯模态复算（N=20, T=0.05, kappa=1）---
N2 = 20; T2 = 0.05
xs = np.arange(N2+1)/N2
XX, YY = np.meshgrid(xs, xs, indexing='ij')
print("\n二维纯模态 1e-3 sin(19pi x)sin(19pi y)，零边界：")
for rnom in (0.25, 0.26):
    M2 = int(round(T2/(rnom*(1/N2)**2)))
    reff = T2/(M2*(1/N2)**2)
    U2 = 1e-3*np.sin(19*np.pi*XX)*np.sin(19*np.pi*YY)
    U2[0, :] = U2[-1, :] = 0.0; U2[:, 0] = U2[:, -1] = 0.0
    for _ in range(M2):
        U2[1:-1, 1:-1] = U2[1:-1, 1:-1] + reff*(U2[2:, 1:-1] + U2[:-2, 1:-1] + U2[1:-1, 2:] + U2[1:-1, :-2] - 4*U2[1:-1, 1:-1])
    g = 1 - 8*reff*np.cos(np.pi/(2*N2))**2
    print(f"  名义 r={rnom}: M={M2} r_eff={reff:.6f} 实算 max|U|={np.max(np.abs(U2)):.4e} 理论 1e-3|g_eff|^M={1e-3*abs(g)**M2:.4e}")

r=0.4: G_39=-0.597533867  高频终点振幅=2.123e-66
r=0.6: G_39=-1.396300800  高频终点振幅=1.573e+11

固定 r=0.4，终点误差 |G_1^M - e^{-0.06pi^2}|：
  J= 20 M=  60 err=9.466413487e-04
  J= 40 M= 240 err=2.359551486e-04
  J= 80 M= 960 err=5.894491079e-05
  J=160 M=3840 err=1.473348857e-05
  说明：上表与 A2、B2 的对应结果一致至舍入容差。
  稳定组终点最大误差 = 2.359551e-04（主要来自低频离散误差）

二维纯模态 1e-3 sin(19pi x)sin(19pi y)，零边界：
  名义 r=0.25: M=80 r_eff=0.250000 实算 max|U|=3.7119e-04 理论 1e-3|g_eff|^M=3.7119e-04
  名义 r=0.26: M=77 r_eff=0.259740 实算 max|U|=1.2883e-01 理论 1e-3|g_eff|^M=1.2883e-01


## 3.3 有限元：弱形式与Galerkin方法（2学时）

| 被审结论 | 验证条件或反例 | 手工结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| A1 §3.3.4.4 / 表 17：置 1、消元、置大数“条件数也一样” | 三个矩阵要分清：$K_{II}$、$\mathrm{diag}(1,K_{II},1)$、含边界耦合的置 $M$ 矩阵 | $K_{II}$ 特征值 $4h^{-1}\sin^2(k\pi/2N)$，$\kappa_2=\cot^2(\pi/2N)$ | $N=8$：$25.274$、$30.782$、$8.21\times10^{11}$；$N=16$：$103.087$、$1.63\times10^{12}$ | “一样”只对内部切片成立，不能推广（B002） |
| 未施加边界时 $K$ 奇异、$\ker K=\{\mathbf 1\}$ | 二次型 $h^{-1}\sum(v_{j+1}-v_j)^2$ | 只在常向量上为零 | 数值零空间维数 1，常向量残差 $\sim10^{-16}$ | 成立；还需 $\mathbf1^TF=0$ 才可能有解 |
| 题 44：置大数后边界值（参考解答 $9.375\times10^{-13}$） | 按解答“替换对角为 $M$、边界右端置零”解 $\delta$ | $\delta=(1-h)/(2(M-1/h))$ | $N=16,M=10^{12}$ 得 $4.6875\times10^{-13}$ | 与参考解答差一倍，需 A1 核查载荷与脚本（C1-03） |
| A2 §3.3：精确载荷时 FEM 节点精确 | 分片线性插值 + 逐单元分部积分 | $I_hu$ 精确满足离散方程，$N=8$ 时节点误差约为机器精度 | $L^2$ 误差 $9.9209\times10^{-3}$、$H^1$ 半范 $2.5118\times10^{-1}$；四点/一点 Gauss 节点误差 $2.24\times10^{-12}$/$6.51\times10^{-3}$ | 成立，但节点精确≠区间逼近好 |

In [5]:
import numpy as np

# --- 1) 刚度矩阵、三个条件数 ---
def KII_of(N):
    h = 1/N; n = N-1
    return (1/h)*(np.diag(2*np.ones(n)) + np.diag(-np.ones(n-1), 1) + np.diag(-np.ones(n-1), -1))

print("刚度矩阵：单元 h^-1[[1,-1],[-1,1]]，内部对角 2/h，端点 1/h")
for N in (8, 16):
    h = 1/N
    KII = KII_of(N)
    lam = np.linalg.eigvalsh(KII)
    K1 = np.zeros((N+1, N+1)); K1[1:N, 1:N] = KII; K1[0, 0] = 1.0; K1[N, N] = 1.0
    Km = np.zeros((N+1, N+1)); Km[1:N, 1:N] = KII; Km[0, 0] = Km[N, N] = 1e12
    Km[0, 1] = Km[1, 0] = Km[N, N-1] = Km[N-1, N] = -1/h
    print(f"N={N}: lambda_min={lam[0]:.6f}, lambda_max={lam[-1]:.6f}")
    print(f"   cond(K_II)={np.linalg.cond(KII, 2):.6f}  cond(置1)={np.linalg.cond(K1, 2):.6f}  cond(置M=1e12)={np.linalg.cond(Km, 2):.4e}")
    print(f"   cot^2(pi/2N)={1/np.tan(np.pi/(2*N))**2:.6f}")

# --- 2) 未施加边界时 K 的零空间 ---
N = 8; h = 1/N
K = np.zeros((N+1, N+1))
for e in range(N):
    K[e:e+2, e:e+2] += (1/h)*np.array([[1, -1], [-1, 1]])
print("\n未施加边界的 K：数值秩 =", np.linalg.matrix_rank(K), "/", N+1,
      " 常向量残差 =", f"{np.max(np.abs(K@np.ones(N+1))):.2e}",
      " 二次型 v^TKv =", f"{np.ones(N+1)@K@np.ones(N+1):.1f}")
print("  -> ker K = span{1}；还需 1^T F = 0 才可能有解")

# --- 3) 题 44 的置大数边界值 ---
N44, h44, M = 16, 1/16, 1e12
delta = (1 - h44)/(2*(M - 1/h44))
print(f"\n题44: delta=(1-h)/(2(M-1/h)) = {delta:.6e}")
print(f"  单边界值 = {delta:.3e}；两边界之和 = {2*delta:.3e}")
print("  参考解答所列 9.375e-13 若为单边界值，则与本式差一倍；若为两边界之和，则口径不同需注明。")

刚度矩阵：单元 h^-1[[1,-1],[-1,1]]，内部对角 2/h，端点 1/h
N=8: lambda_min=1.217927, lambda_max=30.782073
   cond(K_II)=25.274142  cond(置1)=30.782073  cond(置M=1e12)=8.2107e+11
   cot^2(pi/2N)=25.274142
N=16: lambda_min=0.614871, lambda_max=63.385129
   cond(K_II)=103.086869  cond(置1)=103.086869  cond(置M=1e12)=1.6264e+12
   cot^2(pi/2N)=103.086869

未施加边界的 K：数值秩 = 8 / 9  常向量残差 = 0.00e+00  二次型 v^TKv = 0.0
  -> ker K = span{1}；还需 1^T F = 0 才可能有解

题44: delta=(1-h)/(2(M-1/h)) = 4.687500e-13
  单边界值 = 4.688e-13；两边界之和 = 9.375e-13
  参考解答所列 9.375e-13 若为单边界值，则与本式差一倍；若为两边界之和，则口径不同需注明。


In [6]:
import numpy as np
from numpy.polynomial.legendre import leggauss

# --- A2 §3.3 模型：-u''=pi^2 sin(pi x), u(0)=u(1)=0 ---
f  = lambda x: np.pi**2*np.sin(np.pi*x)
ue = lambda x: np.sin(np.pi*x)
due = lambda x: np.pi*np.cos(np.pi*x)

def fem(N, gp):
    h = 1/N; x = np.linspace(0, 1, N+1)
    K = np.zeros((N+1, N+1))
    for e in range(N):
        K[e:e+2, e:e+2] += (1/h)*np.array([[1, -1], [-1, 1]])
    F = np.zeros(N+1)
    Xi, Wi = leggauss(gp)
    for e in range(N):
        s = x[e] + 0.5*h*(Xi + 1)          # 单元内 Gauss 点
        w = 0.5*h*Wi
        for l in range(2):
            phi = (1 - (s - x[e])/h) if l == 0 else ((s - x[e])/h)
            F[e+l] += np.sum(w*f(s)*phi)
    idx = np.arange(1, N)
    U = np.zeros(N+1)
    U[idx] = np.linalg.solve(K[np.ix_(idx, idx)], F[idx])
    # 误差：每单元 10 点 Gauss 独立积分
    X2, W2 = leggauss(10); L2 = 0.0; H1 = 0.0
    for e in range(N):
        s = x[e] + 0.5*h*(X2 + 1); w = 0.5*h*W2
        uh = U[e]*(1 - (s - x[e])/h) + U[e+1]*((s - x[e])/h)
        L2 += np.sum(w*(ue(s) - uh)**2)
        H1 += np.sum(w*(due(s) - (U[e+1]-U[e])/h)**2)
    return np.sqrt(L2), np.sqrt(H1), np.max(np.abs(U - ue(x)))

print("N=8，两种载荷求积：")
for gp in (1, 4):
    L2, H1, nd = fem(8, gp)
    print(f"  {gp} 点 Gauss: |u-uh|_L2={L2:.9e}  |u-uh|_H1={H1:.9e}  节点最大误差={nd:.3e}")

# --- 传统差分基线：节点误差 (pi^2/lambda_h - 1) sin(pi x_i) ---
print("\n差分基线节点最大误差：")
for N in (5, 8, 10, 20, 40):
    h = 1/N; lam = 4/h**2*np.sin(np.pi*h/2)**2
    xs = np.linspace(0, 1, N+1)
    U = (np.pi**2/lam)*np.sin(np.pi*xs); U[0] = U[-1] = 0.0
    print(f"  N={N:2d}: {np.max(np.abs(U-ue(xs))):.9f}")

N=8，两种载荷求积：
  1 点 Gauss: |u-uh|_L2=1.418028045e-02  |u-uh|_H1=2.515927733e-01  节点最大误差=6.513e-03
  4 点 Gauss: |u-uh|_L2=9.920919910e-03  |u-uh|_H1=2.511817694e-01  节点最大误差=2.242e-12

差分基线节点最大误差：
  N= 5: 0.031915927
  N= 8: 0.012950747
  N=10: 0.008265417
  N=20: 0.002058707
  N=40: 0.000514200


## 3.4 AI关联：Neural ODE与PINN（2学时）

| 被审结论 | 验证条件或反例 | 手工结果 | 数值结果 | 判断 |
| --- | --- | --- | --- | --- |
| A1 §3.4.2：残差小即误差小 | 同初值、$f$ 一致 Lipschitz | $|e(t)|\le e^{Lt}|e(0)|+\int_0^t e^{L(t-s)}|r(s)|ds$；配点残差零可误差任意大 | 单配点例：$t=1$ 误差 $e-2=0.71828$ | 需区间控制，B1 反例成立 |
| A1 命题 4 与式 (51)、题 69 的误差分解 | 加减 $u_{\hat\theta}$ 与 $u_{\theta_*}$ | 三项＝离散＋参数偏差＋模型偏差；范数下可能抵消 | 训练节点匹配 $a$：FE $1.3692548146$、RK4 $1.6001828258$（真值 1.6） | 分解写错对象，需统一（C1-04） |
| “两求解器结果之差＝离散误差” | 固定参数、同初值 | 差是两个离散误差之差，只有已知阶或解析参考才能估计单个误差 | $\delta=0.1$: 0.01656955 与 $1.972\times10^{-5}$；闭式 0.03346623 与 $2.086\times10^{-5}$ | 不成立（属已排除的候选） |
| 题 66：固定比值更新三步“拉平”梯度 | $G_r=1$、$G_b=100$ 固定，目标 $\lambda_*=0.01$ | $\lambda_kG_b$ 序列无正平衡点 | 1.585 → 0.02512 → 0.0003981 | 平衡解释不成立（C1-05）；数列本身正确 |
| Poisson：配点残差替代全区间残差 | Green 函数 $G(x,s)$ | $\|e\|_\infty\le\max(|b_0|,|b_1|)+\frac18\|r\|_\infty$ | $A\sin(N\pi x)$ 在每个 $x_i$ 残差为零、中点误差 $|A|$ | 成立，只作反例 |

In [7]:
import numpy as np

# --- 1) 单配点反例：u'=u, u(0)=1, u_theta=1+t ---
print("u'=u, u_theta=1+t：t=0 处残差 =", (1+t*0) - (1+0), " t=1 处误差 =", abs((1+1) - np.e), "= e-2 =", np.e-2)

# --- 2) 训练步长匹配参数的闭式解 ---
a_star, h = 1.6, 0.2
a_fe = (1 - np.exp(-a_star*h))/h
print("\nFE 一步匹配：a_FE = (1-e^{-a*h})/h =", f"{a_fe:.10f}")
def R4s(s): return 1 - s + s**2/2 - s**3/6 + s**4/24
target = np.exp(-a_star*h)
lo, hi = 0.1, 0.5
for _ in range(200):
    mid = 0.5*(lo + hi)
    if R4s(mid) > target: lo = mid
    else: hi = mid
s_root = 0.5*(lo + hi)
a_rk = s_root/h
print("RK4 一步匹配：二分 s∈[0.1,0.5]，a_RK4 =", f"{a_rk:.10f}", " 单调性检查 dR4/ds =", -1 + s_root - s_root**2/2 + s_root**3/6)

# --- 3) 换评估步长后的终点误差 ---
def traj(Rf, a, delta, n):
    y = 1.4
    for _ in range(n): y *= Rf(a*delta)
    return y
print("\n终点绝对误差（真值 1.4e^{-3.2}）：")
for delta in (0.2, 0.1, 0.05):
    n = int(round(2/delta))
    e_fe = abs(traj(lambda z: 1 - z, a_fe, delta, n) - 1.4*np.exp(-3.2))
    e_rk = abs(traj(R4s, a_rk, delta, n) - 1.4*np.exp(-3.2))
    print(f"  delta={delta}: FE {e_fe:.8f}  RK4 {e_rk:.5e}")
print(f"  已学连续模型闭式：FE {abs(1.4*np.exp(-2*a_fe)-1.4*np.exp(-3.2)):.8f}  RK4 {abs(1.4*np.exp(-2*a_rk)-1.4*np.exp(-3.2)):.5e}")

# --- 4) 独立初值 + 101 节点 RMSE ---
ts = np.linspace(0, 2, 101)
for a in (a_fe, a_rk):
    E = np.array([y0*np.exp(-a*ts) - y0*np.exp(-a_star*ts) for y0 in (0.9, 1.4, 2.0)])
    print(f"  a={a:.10f}: RMSE={np.sqrt((E**2).mean()):.8f}")

u'=u, u_theta=1+t：t=0 处残差 = 0.0  t=1 处误差 = 0.7182818284590451 = e-2 = 0.7182818284590451

FE 一步匹配：a_FE = (1-e^{-a*h})/h = 1.3692548146
RK4 一步匹配：二分 s∈[0.1,0.5]，a_RK4 = 1.6001828258  单调性检查 dR4/ds = -0.7257119306782822

终点绝对误差（真值 1.4e^{-3.2}）：
  delta=0.2: FE 0.00000000  RK4 1.38778e-17
  delta=0.1: FE 0.01656955  RK4 1.97227e-05
  delta=0.05: FE 0.02499400  RK4 2.07962e-05
  已学连续模型闭式：FE 0.03346623  RK4 2.08629e-05
  a=1.3692548146: RMSE=0.06538949
  a=1.6001828258: RMSE=0.00004664


In [8]:
import numpy as np
from numpy.polynomial.legendre import leggauss

# --- 1) 题 66 的固定比值更新 ---
lam = 10**-1.8
Gr, Gb = 1.0, 100.0
print("目标 lambda* = Gr/Gb =", Gr/Gb)
for k in range(1, 4):
    print(f"  第{k}步 lambda={lam:.7f}  加权边界梯度 lambda*Gb={lam*Gb:.7f}  相对目标的倍数={lam/0.01:.4f}")
    lam *= 10**-1.8
print("  -> 单调远离平衡；固定比值下不存在正不动点")

# --- 2) Poisson 的 Green 系数 1/8：解析与数值核对 ---
Xi, Wi = leggauss(200)
def green_integral(x):
    return (1-x)*x**2/2 + x*(1-x)**2/2
xsq = np.linspace(0, 1, 10001)
print("\nmax_x int G(x,s)ds =", green_integral(xsq).max(), "  1/8 =", 1/8)
for x0 in (0.3, 0.5, 0.8):
    s1 = 0.5*x0*(Xi+1)                      # [0, x0]，G = s*(1-x0)
    s2 = 0.5*(1-x0)*(Xi+1) + x0             # [x0, 1]，G = x0*(1-s)
    val = 0.5*x0*np.sum(Wi*s1*(1-x0)) + 0.5*(1-x0)*np.sum(Wi*x0*(1-s2))
    print(f"  x={x0}: 数值 {val:.12f}  解析 {green_integral(x0):.12f}")

# --- 3) 有限配点反例（构造性说明）---
print("\n[u'=0, u(0)=1] 取 u_theta(t)=1+A*int_0^t prod_i(s-t_i)^2 ds，t_i 为配点：")
print("  每个配点上 u_theta'≡0，残差全为零；而 t>0 处 |u_theta-1| 随 A 任意增大。")
print("  配点残差不能替代区间残差界。")

目标 lambda* = Gr/Gb = 0.01
  第1步 lambda=0.0158489  加权边界梯度 lambda*Gb=1.5848932  相对目标的倍数=1.5849
  第2步 lambda=0.0002512  加权边界梯度 lambda*Gb=0.0251189  相对目标的倍数=0.0251
  第3步 lambda=0.0000040  加权边界梯度 lambda*Gb=0.0003981  相对目标的倍数=0.0004
  -> 单调远离平衡；固定比值下不存在正不动点

max_x int G(x,s)ds = 0.125   1/8 = 0.125
  x=0.3: 数值 0.105000000000  解析 0.105000000000
  x=0.5: 数值 0.125000000000  解析 0.125000000000
  x=0.8: 数值 0.080000000000  解析 0.080000000000

[u'=0, u(0)=1] 取 u_theta(t)=1+A*int_0^t prod_i(s-t_i)^2 ds，t_i 为配点：
  每个配点上 u_theta'≡0，残差全为零；而 t>0 处 |u_theta-1| 随 A 任意增大。
  配点残差不能替代区间残差界。


## 反例、适用边界与复核结论

**已确认（有独立推导与可复算证据）**
1. C1-01：A1 §3.1.3.2 的阶条件表与 RK4 更新正确，阶段展开的系数错（$y'=t$ 一步 $1/32$ 对 $1/64$）。
2. B001：A1 单元 25 的 Neumann 首行既丢了通量又编错端点，残差 $-0.16$；正确装配下 8 个线性补丁残差约 $2.22\times10^{-16}$。
3. C1-02：一维固定网格的精确门槛是 $[2\cos^2(\pi/2N)]^{-1}$，$r\le1/2$ 只是极限/周期口径。
4. B002 / C1-03：内部块条件数 25.274 与置 1 完整矩阵 30.782 不同；$M=10^{12}$ 时完整矩阵 $8.21\times10^{11}$；题 44 边界值按题设应为 $4.6875\times10^{-13}$。
5. C1-04：误差分解必须固定积分对象；换求解器之差不是单个离散误差。
6. C1-05：固定梯度比值下 $10^{-1.8}$ 连乘无正平衡点，题 66 数列对、解释错。

**适用边界**

- 结论限一维常系数、均匀网格、齐次或线性边界；二维只对等距同尺度、零边界五点格式。
- 有限元结论限分片线性、精确或高阶求积；节点精确不等于 $L^2$/$H^1$ 逼近阶。
- 条件数结论限 2 范数方形矩阵；变系数、非正交对角化不适用。